In [1]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
import json
import os
import pandas as pd
from tensorflow.keras.applications.efficientnet import preprocess_input

# === Constants ===
IMG_SIZE = 224  # Input size for the model

# === Load model ===
model_path = "EfficientNetB0_NoTrain_E30_B32_IMG224.h5"
model = tf.keras.models.load_model(model_path)

# === Load COCO metadata ===
base_path = 'nutritionverse-data'
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')

with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# === Load nutrition data ===
nutrition_data = pd.read_csv("food_nutrients.csv", index_col=0)

# === Label encoder ===
category_ids = [cat['id'] for cat in coco_data['categories']]
label_encoder = LabelEncoder()
label_encoder.fit(category_ids)

# Function to calculate nutrition from detection results
def calculate_nutrition(detection_results):
    total_nutrition = {
        'calories': 0,
        'fat': 0,
        'carbohydrates': 0,
        'protein': 0,
        'calcium': 0,
        'iron': 0,
        'magnesium': 0,
        'potassium': 0,
        'sodium': 0,
        'vitamin_d': 0,
        'vitamin_b12': 0
    }
    
    for result in detection_results:
        food_type = result['category'].lower().replace(' ', '-')
        
        # Check if this food type exists in our nutrition data
        matching_foods = nutrition_data[nutrition_data['food_type'] == food_type]
        
        if not matching_foods.empty:
            food_data = matching_foods.iloc[0]
            # Add each nutrient to the total
            for nutrient in total_nutrition.keys():
                if nutrient in food_data:
                    total_nutrition[nutrient] += food_data[nutrient]
    
    return total_nutrition

# Format detection results as text
def format_detection_results(results):
    if not results:
        return "No food detected"
    
    text_output = "Detected Foods:\n"
    for i, result in enumerate(results, 1):
        text_output += f"{i}. {result['category']} (Confidence: {result['confidence']:.2f})\n"
    
    return text_output

# Function to detect food with optimized batching
def detect_food_in_image(image):
    image = np.array(image.convert("RGB"))
    original_image = image.copy()
    results = []

    window_sizes = [
        (image.shape[1] // 2, image.shape[0] // 2),
        (image.shape[1] // 3, image.shape[0] // 3),
        (image.shape[1] // 4, image.shape[0] // 4)
    ]
    step_divisors = [2, 4, 8, 12]

    all_windows = []
    all_coords = []

    for win_w, win_h in window_sizes:
        if win_w < 50 or win_h < 50:
            continue
        for divisor in step_divisors:
            step_size = max(1, min(image.shape[0], image.shape[1]) // divisor)
            for y in range(0, image.shape[0] - win_h + 1, step_size):
                for x in range(0, image.shape[1] - win_w + 1, step_size):
                    window = original_image[y:y+win_h, x:x+win_w]
                    resized = cv2.resize(window, (IMG_SIZE, IMG_SIZE))
                    input_img = preprocess_input(resized.astype(np.float32))
                    all_windows.append(input_img)
                    all_coords.append((x, y, win_w, win_h))

    if not all_windows:
        return Image.fromarray(original_image), "No food detected", {}

    # Batch predictions
    all_windows = np.stack(all_windows)
    batch_size = 64
    predictions = []
    for i in range(0, len(all_windows), batch_size):
        batch = all_windows[i:i+batch_size]
        preds = model.predict(batch, verbose=0)
        predictions.extend(preds)

    sliding_window_results = []
    for pred, (x, y, w, h) in zip(predictions, all_coords):
        class_idx = np.argmax(pred)
        confidence = pred[class_idx]
        if confidence < 0.9:
            continue
        predicted_class = label_encoder.inverse_transform([class_idx])[0]
        category_name = next((cat['name'] for cat in coco_data['categories'] if cat['id'] == predicted_class), "Unknown")
        sliding_window_results.append({
            'class_idx': class_idx,
            'category': category_name,
            'confidence': float(confidence),
            'bbox': [x, y, w, h]
        })

    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    keep_results = []

    for current in sliding_window_results:
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        overlaps = False
        for kept in keep_results:
            x3, y3, w2, h2 = kept['bbox']
            x4, y4 = x3 + w2, y3 + h2
            inter_x = max(0, min(x2, x4) - max(x1, x3))
            inter_y = max(0, min(y2, y4) - max(y1, y3))
            intersection = inter_x * inter_y
            union = w1 * h1 + w2 * h2 - intersection
            iou = intersection / union if union > 0 else 0
            if iou > 0.1 and current['category'] == kept['category']:
                overlaps = True
                break
        if not overlaps:
            keep_results.append(current)

    # Process the detection results without drawing boxes
    for result in keep_results:
        x, y, w, h = result['bbox']
        category_name = result['category']
        confidence = result['confidence']
        cv2.rectangle(original_image, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.putText(original_image, f"{category_name} ({confidence:.2f})",
                    (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        results.append({
            'category': result['category'],
            'confidence': result['confidence']
        })
    
    # Format the detection results as text
    detection_text = format_detection_results(results)
    
    # Calculate nutrition information from detected foods
    nutrition_info = calculate_nutrition(results)
    
    return Image.fromarray(original_image), detection_text, nutrition_info



# === Gradio Interface ===
interface = gr.Interface(
    fn=detect_food_in_image,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Original Image"),
        gr.Textbox(label="Detected Foods"),
        gr.JSON(label="Nutritional Information")
    ],
    title="Food Detection with Nutritional Analysis",
    description="Upload an image to detect food using a custom trained EfficientNetB0 model. The system will identify foods and calculate their total nutritional content.",
    flagging_mode="never"
)

# === Launch ===
interface.launch(share=True)

2025-05-30 18:17:43.697534: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748629063.710612  434024 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748629063.714764  434024 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748629063.726812  434024 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748629063.726826  434024 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748629063.726829  434024 computation_placer.cc:177] computation placer alr

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d790c92f0f87c774e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
